# The pedigree packages in Python

This notebook uses the workbench's Python packages directly. They are the same packages that the **Tools** tab and
the **Terminal** run. Run the cells in order from the top: click a cell and press **Shift+Enter**. The first code cell
starts Python in your browser, which takes a few seconds the first time.

The example files are in `/data/examples`, and can't be changed. Anything you make goes into your own folder, `~`
(`/home/user`). On your own computer the same code works once you've installed the packages with
`pip install studio2ped studio2ga4gh …`, as the versions on PyPI are the ones that run here.

## A Pedigree Studio drawing

A drawing is a JSON file holding the people (with their symbols, fills and positions), the couples, the children, the
twins and the colour key. `studio2ga4gh` reads it into Python objects and works out the pedigree numbers (I-1, II-3 …)
just as the Studio shows them.

In [ ]:
import json
from studio2ga4gh import parse_session

data = json.load(open("/data/examples/okafor.studio.json"))
drawing = parse_session(data)
numbers = drawing.numbers()
people = [p for p in drawing.persons.values() if not p.is_marker]
print(len(people), "people; the colour key:", drawing.legend)
for p in sorted(people, key=lambda p: numbers[p.id])[:6]:
    print(numbers[p.id], p.sex, "affected" if p.filled else "", drawing.condition_names(p))

## Drawing → GA4GH Pedigree and Phenopackets Family

The conditions in the key need ontology terms. Here they come from a terms file, which the **Match conditions to
terms** tool in the Tools tab can make for you. A condition without a term is left out, and a note says so.

In [ ]:
import os
from studio2ga4gh import convert_file

os.makedirs("/home/user/results", exist_ok=True)
terms = json.load(open("/data/examples/example-terms.json"))
result = convert_file("/data/examples/okafor.studio.json", "okafor-nb", "/home/user/results", terms=terms)
print(result.summary)
for path, _ in result.files:
    print("wrote", path)

## Checking a file against the standard

`pedigree_ga4gh` is the GA4GH implementation that studio2ga4gh, ga4gh2studio and the phenopacket toolkit share.
`validate` applies the checks of the
Phenopackets v2 schema and of phenopacket-tools, and those of the GA4GH Pedigree, such as the KIN relations, the ids
and the parents.

In [ ]:
import pedigree_ga4gh as pg

doc = pg.load("/home/user/results/okafor-nb_ga4gh-pedigree.json")
findings = pg.validate(doc)
print(pg.render_findings(findings))

old = pg.load("/data/examples/older/studio2ga4gh-0.3_ga4gh-pedigree.json")
n = len(pg.errors(pg.validate(old)))
print(n, "error" if n == 1 else "errors", "in a file from studio2ga4gh 0.3")

## The family as Python objects

`from_document` reads any of the GA4GH forms into a `Kindred`, which holds the family members with their terms and
the relations between them, named with KIN's kinship terms.

In [ ]:
kindred, notes = pg.from_document(doc)
proband = [m for m in kindred.members if m.index][0]
father, mother, _ = kindred.genetic_parents(proband.id)
print("proband", proband.id, "– father", father, "– mother", mother)
for r in kindred.relations[:5]:
    print(r.individual, pg.kin.get(r.kin).label, r.relative)

## The phenopacket toolkit

`phenopacket_toolkit` reads every form (Phenopacket, Family, Cohort and GA4GH Pedigree) and does the same jobs as
`phenotk`: viewing, statistics, filtering, de-identification and conversion.

In [ ]:
import phenopacket_toolkit as ptk
from phenopacket_toolkit.ops import render, deidentify

family = ptk.read("/data/examples/okafor_phenopackets-family.json")
print(render(family))

shared = deidentify(family, remap_ids="seq")
print("new ids:", shared.crosswalk)

## PED files

`mpedtools` reads PED files and MPED files, which hold several conditions, each in its own column.

In [ ]:
import mpedtools

ped = mpedtools.read("/data/examples/okafor.mped")
print(mpedtools.summarise(ped))
print([issue.message for issue in mpedtools.validate(ped)])

## The commands, from a cell

A line that starts with `!` runs a command, as it would in the terminal. The pedigree programs run here in the
notebook's own Python, so a plain `!` line works. For pipes (`|`) and `>`, use the Terminal tab instead.

In [ ]:
!studio2ped /data/examples/okafor.studio.json -d ~/results -o okafor-nb
!mpedtools info ~/results/okafor-nb.mped